In [1]:
import wandb
from wandb.apis.public.runs import Run
import math
from typing import Any
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
api = wandb.Api()

In [3]:
runs = api.runs(
    "graphcore/llama-mobile",
    filters={"config.name": "no-vision-16-12-25", "state": "finished"},
)

In [8]:
# NOTE: All runs use INT3 element dtype
def cleanup_run(run: Run) -> dict[str, Any] | None:
    d = {}
    d["freeze_vision_params"] = run.config["training"]["freeze_params"]
    d["n_steps"] = run.config["training"]["n_steps"]
    d["lr_exponent"] = math.log2(run.config["training"]["optimiser"]["lr"])
    d["vqa"] = run.summary["downstream.vqa.accuracy"]
    d["ai2d"] = run.summary["downstream.ai2d.accuracy"]
    d["chartqa"] = run.summary["downstream.chartqa.accuracy"]
    d["docvqa"] = run.summary["downstream.docvqa.anls"]

    # d["vqa_easy"] = run.summary["downstream.vqa.accuracy_easy"]
    history = run.history()
    d["train_loss"] = (
        np.mean(history["train/loss"].tail(10).values)
        if "train/loss" in history
        else None
    )
    d["val_loss"] = run.summary["val/loss"] if "val/loss" in run.summary else None

    return d

In [9]:
df = pd.DataFrame([cleanup_run(run) for run in runs])

In [12]:
df.sort_values(by=["n_steps", "lr_exponent"], ascending=True, inplace=True)
# df = df.drop_duplicates()

In [13]:
df

,freeze_vision_params,n_steps,lr_exponent,vqa,ai2d,chartqa,docvqa,train_loss,val_loss
20,"[vision_model, multi_modal_projector]",256,-19.0,0.640299,0.325195,0.592773,0.389012,0.092156,0.136966
25,"[vision_model, multi_modal_projector]",256,-19.0,0.629232,0.261719,0.580078,0.368913,0.093230,0.145876
12,"[vision_model, multi_modal_projector]",256,-18.0,0.675130,0.196289,0.540039,0.319186,0.079797,0.116541
13,"[vision_model, multi_modal_projector]",256,-18.0,0.680990,0.255859,0.554688,0.324327,0.078834,0.111411
6,"[vision_model, multi_modal_projector]",256,-17.0,0.671549,0.235352,0.334961,0.392693,0.074654,0.102923
9,"[vision_model, multi_modal_projector]",256,-16.0,0.445964,0.245117,0.231445,0.353408,0.068821,0.147834
10,"[vision_model, multi_modal_projector]",256,-16.0,0.300781,0.281250,0.176758,0.212638,0.070746,0.177238
31,[],256,-15.0,0.589844,0.192383,0.161133,0.092331,0.088498,0.147995
21,"[vision_model, multi_modal_projector]",512,-19.0,0.639323,0.208984,0.658203,0.346225,0.065448,0.122187
22,"[vision_model, multi_modal_projector]",512,-19.0,0.610026,0.183594,0.655273,0.309326,0.065808,0.129310


In [ ]:
# Baseline (bflo16) results
from pathlib import Path
import json
from eval import vqa

path = Path().resolve().parent / "20251021-DownstreamTest/out/vqa/2025-10-21T17-49-27"
baseline = {}
for task_name in ["ai2d", "chartqa", "docvqa", "vqa"]:
    results = []
    with open(path / f"{task_name}.jsonl") as f:
        for line in f:
            results.append(json.loads(line))
    metric = vqa.TASKS[task_name].METRICS[0]
    out = [x[metric] for x in results]
    baseline[task_name] = sum(out) / len(out)

In [ ]:
tasks = ["vqa", "chartqa", "docvqa", "ai2d"]
for task in tasks:
    g = sns.relplot(
        data=df,
        x="n_steps",
        y=task,
        hue="mode",
        col="lr_exponent",
        height=3
    )
    # step_0 = df.loc[df["n_steps"] == 0, task].iloc[0]
    for ax in g.axes.flat:
        # ax.axhline(step_0, linestyle="--", color="black")
        ax.axhline(baseline[task], linestyle="--", color="black")

In [ ]:
sns.relplot(
    data=df, x="lr_exponent", y="train_loss", col="n_steps", hue="mode", height=3
)

In [ ]:
sns.relplot(data=df, x="lr_exponent", y="val_loss", col="n_steps", hue="mode", height=3)